# Zero shot

Zero-shot text-to-tracklet retrieval on TVPReid test. The new frame models run first: SigLIP 2 So400m, Perception Encoder L/14, and IRRA (CUHK-PEDES). Then InternVideo2-1B-s2, X-CLIP, LanguageBind, and InternVideo2 CLIP-S.

Clones the shawaf repo and OpenGVLab/InternVideo. Enable GPU and Internet. The 1B-s2 checkpoint is gated: accept the license on Hugging Face, then attach a Kaggle secret named `hugging_face` with your token. The install cell loads that secret. IRRA weights download from the official Google Drive link.

SigLIP 2, Perception Encoder, IRRA, CLIP-S, X-CLIP, and LanguageBind use 8-frame clips. Frame models average the frames inside each clip, then the window pools run as usual. Perception Encoder truncates text at 32 tokens. IRRA resizes every frame to 384x128. 1B-s2 uses 4-frame clips. InternVideo2-CLIP-1B is not in this list: the Hub file is a 14 MB add-on and the text tower is a 25 GB checkpoint, which does not fit a T4.


In [1]:
from pathlib import Path

MODELS = [
    "siglip2",
    "pe_core_l14",
    "irra",
    "internvideo2_s2_1b",
    "xclip",
    "languagebind",
    "internvideo2",
]
SUBSETS = ["prid", "ilids", "duke"]
BATCH = 4
TEXT_BATCH = 16
DEVICE = "cuda"
SPLIT = "test"
POOLS = ("mean", "mean_s8", "max", "query_max")

# 8-frame encoders. 1B-s2 is built for 4 frames, so its window length is 4.
CLIP8 = {"siglip2", "pe_core_l14", "irra", "xclip", "languagebind", "internvideo2"}
NATIVE_FRAMES = {"internvideo2_s2_1b": 4}
MODEL_BATCH = {
    "siglip2": 1,
    "pe_core_l14": 1,
    "irra": 2,
    "internvideo2_s2_1b": 1,
    "internvideo2": 2,
    "languagebind": 2,
}

SINGLE_PROTOCOLS = [
    {"name": "uniform8", "num_frames": 8, "frame_sample": "uniform", "models": "clip8"},
    {"name": "uniform4", "num_frames": 4, "frame_sample": "uniform", "models": "s2"},
    {"name": "middle4", "num_frames": 4, "frame_sample": "middle", "models": "s2"},
]
WINDOW_PROTOCOLS = [
    {"name": "vt_1fps_n12", "sample_fps": 1.0, "max_frames": 12, "stride": 4},
    {"name": "vt_2fps_n32", "sample_fps": 2.0, "max_frames": 32, "stride": 4},
    {"name": "reid_8fps_n64", "sample_fps": 8.0, "max_frames": 64, "stride": 4},
]

RESULTS_DIR = Path("/kaggle/working/zero_shot_results") if Path("/kaggle/working").is_dir() else Path("results/zero_shot")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Zero shot", MODELS, SUBSETS, flush=True)


Zero shot ['siglip2', 'pe_core_l14', 'irra', 'internvideo2_s2_1b', 'xclip', 'languagebind', 'internvideo2'] ['prid', 'ilids', 'duke']


In [2]:
import os
import subprocess
import sys
from pathlib import Path

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
INTERNVIDEO_URL = "https://github.com/OpenGVLab/InternVideo.git"
EXPECTED = "0.1.16"


def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    intern_dir = Path("/kaggle/working/InternVideo")
    if shawaf_dir.exists():
        run(["git", "fetch", "origin"], cwd=shawaf_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=shawaf_dir)
    else:
        run(["git", "clone", SHAWAF_URL, str(shawaf_dir)])
    if intern_dir.exists():
        run(["git", "fetch", "origin"], cwd=intern_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=intern_dir)
    else:
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]"])
    run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "einops", "timm", "av", "imageio", "librosa", "soundfile",
            "pandas", "pyyaml", "scipy", "wandb", "gdown",
        ]
    )
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "decord"])
    except subprocess.CalledProcessError:
        print("decord wheel failed; the fine-tune path stubs it and uses PyAV", flush=True)
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break
    intern_dir = shawaf_dir / "InternVideo"
    if not intern_dir.is_dir():
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)], cwd=shawaf_dir)

os.environ["INTERNVIDEO_ROOT"] = str(intern_dir)
if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))

for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("INTERNVIDEO_ROOT", os.environ["INTERNVIDEO_ROOT"], flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and run this cell again after origin/main updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token and Path("/kaggle/working").is_dir():
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("hugging_face")
if not token:
    raise RuntimeError(
        "Hugging Face token missing. On Kaggle, add a secret named hugging_face "
        "and attach it to this notebook."
    )
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
from huggingface_hub import login

login(token=token, add_to_git_credential=False)
print("Hugging Face token loaded", flush=True)


+ git clone https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git /kaggle/working/shawaf-vlm


Cloning into '/kaggle/working/shawaf-vlm'...


+ git clone https://github.com/OpenGVLab/InternVideo.git /kaggle/working/InternVideo


Cloning into '/kaggle/working/InternVideo'...


+ /usr/bin/python3 -m pip install -q -e /kaggle/working/shawaf-vlm[all]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
+ /usr/bin/python3 -m pip install -q einops timm av imageio librosa soundfile pandas pyyaml scipy wandb gdown
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 52.2 MB/s eta 0:00:00
+ /usr/bin/python3 -m pip install -q decord
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 80.1 MB/s eta 0:00:00
shawaf_vlm 0.1.16 /kaggle/working/shawaf-vlm/shawaf_vlm/__init__.py
INTERNVIDEO_ROOT /kaggle/working/InternVideo


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face token loaded


In [3]:
from shawaf_vlm.data.tvpreid import download_tvpreid, load_tvpreid

DATA_ROOT = download_tvpreid(configs=tuple(SUBSETS), split=SPLIT)
print("TVPReid", SPLIT, DATA_ROOT, flush=True)


Hub repo : bassatbassat/TVPReid
Split    : test only (skips train/val mp4s)
Subsets  : prid, ilids, duke


Metadata:   0%|          | 0/3 [00:00<?, ?file/s]

Videos   : 820 files


Videos test:   0%|          | 0/820 [00:00<?, ?file/s]

TVPReid test /kaggle/working/TVPReid


In [4]:
import gc
import json

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet, evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

METRIC_KEYS = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]


def _jsonable(metrics):
    out = {}
    for key in METRIC_KEYS:
        if key not in metrics:
            continue
        value = metrics[key]
        out[key] = float(value) if value is not None else None
    return out


def _record(model, subset, protocol, pool, sampling, metrics):
    row = {
        "model": model,
        "subset": subset,
        "protocol": protocol,
        "pool": pool,
        "sampling": sampling,
    }
    row.update(_jsonable(metrics))
    rows.append(row)
    safe = protocol.replace("/", "-")
    out = RESULTS_DIR / f"{model}_tvpreid_{subset}_{safe}_{pool}.json"
    out.write_text(json.dumps(row, indent=2), encoding="utf-8")
    print(format_metrics(metrics), flush=True)
    print("Wrote", out, flush=True)


ensure_cuda_healthy(DEVICE)
rows = []
frame_cache = DATA_ROOT / "frame_cache"

for name in MODELS:
    native = NATIVE_FRAMES.get(name, 8)
    batch = MODEL_BATCH.get(name, BATCH)
    encoder = build_encoder(name, device=DEVICE)
    family = "clip8" if name in CLIP8 else "s2"
    for subset in SUBSETS:
        splits = load_tvpreid(subset, split=SPLIT, root=DATA_ROOT)
        print(
            f"{name} {subset}: {len(splits.query)} queries, {len(splits.gallery)} gallery",
            flush=True,
        )
        for spec in SINGLE_PROTOCOLS:
            if spec["models"] != family:
                continue
            sampling = f"{spec['frame_sample']} {spec['num_frames']} frames"
            print(f"== {name} {subset} {spec['name']} ({sampling})", flush=True)
            metrics = evaluate_text_to_tracklet(
                encoder,
                splits,
                num_frames=spec["num_frames"],
                batch_size=batch,
                text_batch_size=TEXT_BATCH,
                junk_same_camera=False,
                frame_cache=frame_cache,
                frame_sample=spec["frame_sample"],
            )
            _record(name, subset, spec["name"], "none", sampling, metrics)
        for spec in WINDOW_PROTOCOLS:
            sampling = (
                f"sliding windows of {native}, stride {spec['stride']}, "
                f"{spec['sample_fps']:g} fps, cap {spec['max_frames']} frames"
            )
            print(f"== {name} {subset} {spec['name']} ({sampling})", flush=True)
            pooled = evaluate_text_to_tracklet_windows(
                encoder,
                splits,
                num_frames=native,
                stride=spec["stride"],
                sample_fps=spec["sample_fps"],
                max_frames=spec["max_frames"],
                pools=POOLS,
                batch_size=batch,
                text_batch_size=TEXT_BATCH,
                junk_same_camera=False,
                frame_cache=frame_cache,
            )
            for pool, metrics in pooled.items():
                print(f"-- pool {pool}", flush=True)
                _record(name, subset, spec["name"], pool, sampling, metrics)
    del encoder
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

print(f"Collected {len(rows)} rows", flush=True)


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model device: cuda:0 dtype=torch.float16 requested=cuda (Tesla T4)
siglip2 prid: 284 queries, 142 gallery
== siglip2 prid uniform8 (uniform 8 frames)


Decode frames:   0%|          | 0/142 [00:00<?, ?video/s]

siglip2 texts: 100%|██████████| 18/18 [00:00<00:00, 19.15batch/s]

Text-to-tracklet evaluation
  Rank-1  : 33.45
  Rank-5  : 68.31
  Rank-10 : 79.58
  Rank-20 : 89.44
  Rank-50 : 98.59
  mAP     : 48.64
  MdR     : 3.00
  MnR     : 7.60
  nDCG@10 : 55.28
  mINP    : 48.64
  valid Q : 284
  gallery : 142
  time    : decode 3.3s  video 56.7s  text 0.9s  score 0.1s  total 60.9s
  video   : 399.2 ms/item
  GPU     : peak 2.23 GB allocated, 2.33 GB reserved
Wrote /kaggle/working/zero_shot_results/siglip2_tvpreid_prid_uniform8_none.json
== siglip2 prid vt_1fps_n12 (sliding windows of 8, stride 4, 1 fps, cap 12 frames)


...

-- pool mean
Text-to-tracklet evaluation
  Rank-1  : 17.74
  Rank-5  : 33.00
  Rank-10 : 43.53
  Rank-20 : 55.22
  Rank-50 : 68.41
  mAP     : 26.06
  MdR     : 15.50
  MnR     : 56.86
  nDCG@10 : 28.98
  mINP    : 26.06
  valid Q : 1206
  gallery : 603
  clips   : 2561
  time    : decode 0.1s  video 905.7s  text 1.0s  score 0.5s  total 907.4s
  video   : 353.7 ms/item
  GPU     : peak 1.54 GB allocated, 1.58 GB reserved
Wrote /kaggle/working/zero_shot_results/internvideo2_tvpreid_duke_vt_2fps_n32_mean.json
-- pool mean_s8
Text-to-tracklet evaluation
  Rank-1  : 17.66
  Rank-5  : 33.33
  Rank-10 : 43.03
  Rank-20 : 54.56
  Rank-50 : 68.74
  mAP     : 26.07
  MdR     : 16.00
  MnR     : 56.07
  nDCG@10 : 28.88
  mINP    : 26.07
  valid Q : 1206
  gallery : 603
  clips   : 2561
  time    : decode 0.1s  video 905.7s  text 1.0s  score 0.5s  total 907.4s
  video   : 353.7 ms/item
  GPU     : peak 1.54 GB allocated, 1.58 GB reserved
Wrote /kaggle/working/zero_shot_results/internvideo2_tvprei

Decode windows:   0%|          | 0/603 [00:00<?, ?video/s]

Window encode: 603 tracklets, 8522 clips (window=8 stride=4 fps=8 max=64)


internvideo2 texts: 100%|██████████| 76/76 [00:01<00:00, 70.30batch/s]


-- pool mean
Text-to-tracklet evaluation
  Rank-1  : 17.33
  Rank-5  : 34.58
  Rank-10 : 44.11
  Rank-20 : 55.89
  Rank-50 : 69.40
  mAP     : 26.44
  MdR     : 15.00
  MnR     : 56.11
  nDCG@10 : 29.45
  mINP    : 26.44
  valid Q : 1206
  gallery : 603
  clips   : 8522
  time    : decode 0.7s  video 3012.0s  text 1.1s  score 0.5s  total 3014.3s
  video   : 353.4 ms/item
  GPU     : peak 1.54 GB allocated, 1.58 GB reserved
Wrote /kaggle/working/zero_shot_results/internvideo2_tvpreid_duke_reid_8fps_n64_mean.json
-- pool mean_s8
Text-to-tracklet evaluation
  Rank-1  : 17.08
  Rank-5  : 34.99
  Rank-10 : 44.36
  Rank-20 : 55.97
  Rank-50 : 68.82
  mAP     : 26.36
  MdR     : 15.00
  MnR     : 55.95
  nDCG@10 : 29.47
  mINP    : 26.36
  valid Q : 1206
  gallery : 603
  clips   : 8522
  time    : decode 0.7s  video 3012.0s  text 1.1s  score 0.5s  total 3014.3s
  video   : 353.4 ms/item
  GPU     : peak 1.54 GB allocated, 1.58 GB reserved
Wrote /kaggle/working/zero_shot_results/internvideo2_

In [5]:
import pandas as pd

TABLE_COLUMNS = [
    "model", "subset", "protocol", "pool", "sampling",
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]
table = pd.DataFrame(rows)
for column in TABLE_COLUMNS:
    if column not in table.columns:
        table[column] = pd.NA
table = table[TABLE_COLUMNS].sort_values(
    ["model", "subset", "protocol", "pool"],
    kind="mergesort",
)
table_path = RESULTS_DIR / "zero_shot_table.csv"
table.to_csv(table_path, index=False)

pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 80)
float_cols = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]
shown = table.copy()
for column in float_cols:
    shown[column] = pd.to_numeric(shown[column], errors="coerce").round(2)
print(shown.to_string(index=False))
print("Wrote", table_path, flush=True)

best = (
    table.sort_values("Rank-1", ascending=False, kind="mergesort")
    .groupby(["model", "subset"], as_index=False)
    .head(1)
)
print("\nBest Rank-1 per model and subset")
print(
    best[["model", "subset", "protocol", "pool", "sampling", "Rank-1", "mAP", "MdR"]]
    .to_string(index=False)
)


             model subset      protocol      pool                                             sampling  Rank-1  Rank-5  Rank-10  Rank-20  Rank-50   mAP   MdR    MnR  nDCG@10  mINP  num_valid_queries  num_gallery  num_clips  decode_s  video_s  text_s  score_s  total_s  video_ms_per_item  peak_gpu_gb  reserved_gpu_gb
      internvideo2   duke reid_8fps_n64       max sliding windows of 8, stride 4, 8 fps, cap 64 frames   14.59   28.77    35.49    45.94    62.02 21.99  25.5  68.88    23.99 21.99             1206.0        603.0     8522.0      0.66  3012.01    1.09     0.53  3014.28             353.44         1.54             1.58
      internvideo2   duke reid_8fps_n64      mean sliding windows of 8, stride 4, 8 fps, cap 64 frames   17.33   34.58    44.11    55.89    69.40 26.44  15.0  56.11    29.45 26.44             1206.0        603.0     8522.0      0.66  3012.01    1.09     0.52  3014.27             353.44         1.54             1.58
      internvideo2   duke reid_8fps_n64   mean_s8

## Results

Tesla T4, TVPReid test. Each accuracy cell is **Rank-1 / mAP / MdR** for that model's best Rank-1 on the subset. Milliseconds per clip and peak GPU are stable across protocols. Duke video encode is the wall clock of the run that set that model's best Duke Rank-1.

### Rank-1 / mAP / median rank

| Model | PRID | iLIDS | Duke |
|---|---:|---:|---:|
| IRRA | **66.55** / 76.09 / 1 | **30.00** / 41.27 / 5 | **38.06** / 50.79 / 2 |
| Perception Encoder L/14 | 38.38 / 52.04 / 2 | 19.33 / 32.55 / 6 | 17.66 / 28.12 / 11 |
| SigLIP 2 So400m | 35.56 / 50.84 / 3 | 18.67 / 28.88 / 10 | 21.97 / 33.03 / 7 |
| InternVideo2 CLIP-S | 32.39 / 43.77 / 4 | 13.33 / 24.14 / 11.5 | 18.49 / 28.36 / 12 |
| LanguageBind | 16.20 / 24.82 / 16 | 12.67 / 22.09 / 14 | 7.30 / 14.46 / 40.5 |
| X-CLIP | 4.93 / 10.05 / 43.5 | 5.33 / 10.52 / 38.5 | 1.41 / 3.95 / 164 |
| InternVideo2-1B-s2 | 4.23 / 10.78 / 34.5 | 4.00 / 10.26 / 29 | 1.82 / 3.82 / 159 |

### Setting, speed, and GPU

| Model | Best setting | ms / clip | Peak GPU | Duke video encode |
|---|---|---:|---:|---:|
| IRRA | PRID 8 fps `mean_s8`; iLIDS 2 fps `mean`; Duke 8 fps `query_max` | 34 | 0.31 GB | 4.8 min |
| Perception Encoder L/14 | PRID 8 fps `mean_s8`; iLIDS and Duke 8 fps `query_max` | 238 | 1.32 GB | 33.8 min |
| SigLIP 2 So400m | 8 fps `query_max` on all three subsets | 421 | 2.23 GB | 59.8 min |
| InternVideo2 CLIP-S | PRID 2 fps `max`; iLIDS and Duke 8 fps `query_max` | 354 | 1.54 GB | 50.2 min |
| LanguageBind | PRID 2 fps `mean`; iLIDS and Duke 8 fps `mean` | 192 | 1.22 GB | 27.2 min |
| X-CLIP | PRID and Duke 8 fps `mean`; iLIDS 2 fps `mean` | 30 | 0.45 GB | 4.2 min |
| InternVideo2-1B-s2 | PRID uniform 4; iLIDS 2 fps `query_max`; Duke 1 fps `mean_s8` | 262 | 2.81 GB | 6.9 min |